# Chapter 13: Streamlit Basics

Everything you've built so far only shows results as text in a notebook. Streamlit turns a plain Python script into a page you can open in a web browser, with very little extra code.

## 1. Installing Streamlit

Streamlit is a separate tool, installed the same way as `requests` or `google-genai` — with `pip install streamlit`.

**Task:** Open a terminal, run `pip install streamlit`, then run `streamlit --version` to confirm it installed correctly.

## 2. Starting Your App File

A Streamlit app isn't written inside this notebook — it lives in its own `.py` file inside a folder named `my_app`, which Streamlit turns into a webpage.

**Task:** Create a folder named `my_app`, and inside it create a file named `app.py` with this content:

```python
import streamlit as st
```

## 3. Giving Your App a Title

`st.title()` displays large heading text at the top of the page, similar to `print()` but shown in the browser instead of the notebook.

**Task:** Update `my_app/app.py` to look like this:

```python
import streamlit as st

st.title("My First App")
```

## 4. Running Your App

Typing `streamlit run my_app/app.py` in a terminal starts a small local web server and opens the page in your browser. Every time you interact with the page, Streamlit reruns the whole script from top to bottom.

**Task:** Open a terminal in this folder and run `streamlit run my_app/app.py`. Check your browser for the title you just added.

## 5. Displaying Text with st.write()

`st.write()` is Streamlit's all-purpose display tool — give it text, a number, or a dictionary, and it shows up on the page.

**Task:** Update `my_app/app.py` to look like this, save the file, then check your browser (Streamlit will offer to rerun):

```python
import streamlit as st

st.title("My First App")
st.write("Welcome!")
```

## 6. Getting Typed Text with st.text_input()

`st.text_input()` draws a text box on the page and gives back whatever the visitor typed into it.

**Task:** Update `my_app/app.py` to look like this:

```python
import streamlit as st

st.title("My First App")
name = st.text_input("What's your name?")
st.write("Hello,", name)
```

## 7. A Chat-Style Box with st.chat_input()

`st.chat_input()` is a text box built for chat apps — it stays pinned to the bottom of the page and gives back `None` until the visitor types something and presses enter.

**Task:** Update `my_app/app.py` to look like this:

```python
import streamlit as st

st.title("My First App")
question = st.chat_input("Ask a question")
st.write(question)
```

## 8. Showing a Chat Bubble with st.chat_message()

`with st.chat_message("user"):` wraps whatever runs inside it in a speech-bubble style block, with an icon showing who "said" it — `"user"` or `"assistant"`.

**Task:** Update `my_app/app.py` to look like this:

```python
import streamlit as st

st.title("My First App")
question = st.chat_input("Ask a question")

if question:
    with st.chat_message("user"):
        st.write(question)
```

## 9. Loading the Gemini API Key Inside Your App

`app.py` is a separate script from this notebook, so it needs its own `load_dotenv()` and `client` before it can call Gemini. It reads the same `.env` file used earlier, as long as you run `streamlit run` from this folder.

**Task:** Update `my_app/app.py` to look like this:

```python
import os
from dotenv import load_dotenv
import streamlit as st
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

st.title("My First App")
question = st.chat_input("Ask a question")

if question:
    with st.chat_message("user"):
        st.write(question)
```

## 10. Putting It Together: One Question, One Answer

Let's finish the app: send the question to Gemini, and show the reply in its own chat bubble. Each question gets a fresh answer — nothing from earlier questions is remembered.

**Task:** Update `my_app/app.py` one last time to look like this:

```python
import os
from dotenv import load_dotenv
import streamlit as st
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

st.title("My First App")
question = st.chat_input("Ask a question")

if question:
    with st.chat_message("user"):
        st.write(question)

    result = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=question,
    )

    with st.chat_message("assistant"):
        st.write(result.text)
```

## 11. Remembering Past Messages with Session State

Streamlit reruns the whole script from top to bottom on every interaction, so anything not deliberately saved disappears — including earlier questions and answers. `st.session_state` is a dictionary-like object that survives reruns, so a list stored there sticks around.

**Task:** Update `my_app/app.py` to look like this: store each question and answer as a dictionary in `st.session_state.messages`, and loop over that list near the top of the script to redisplay every past message before handling a new one.

```python
import os
from dotenv import load_dotenv
import streamlit as st
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

st.title("My First App")

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])

question = st.chat_input("Ask a question")

if question:
    with st.chat_message("user"):
        st.write(question)
    st.session_state.messages.append({"role": "user", "content": question})

    result = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=question,
    )

    with st.chat_message("assistant"):
        st.write(result.text)
    st.session_state.messages.append({"role": "assistant", "content": result.text})
```

## 12. Seeing It Run

**Task:** In a terminal, run `streamlit run my_app/app.py` again. Ask a couple of questions, one after another — each new question and answer should appear as chat bubbles, with your earlier ones still visible above.

## Recap

- Streamlit turns a plain `.py` script into a webpage, started with `streamlit run my_app/app.py` in a terminal.
- The whole script reruns from top to bottom every time someone interacts with the page.
- `st.title()` and `st.write()` display headings and general content; `st.text_input()` and `st.chat_input()` collect what a visitor types.
- `st.chat_message("user")` and `st.chat_message("assistant")` wrap content in labeled chat bubbles.
- A Streamlit app is its own script, so it needs its own imports, `load_dotenv()`, and Gemini client, just like the ones set up in an earlier notebook.
- Combining a `chat_input()` box with one `client.models.generate_content()` call is enough for a simple one-question, one-answer chat page.
- `st.session_state` survives reruns, so storing each message there and redisplaying the list keeps past questions and answers visible on the page.